# XiloScan · Etapa 2 — Modelo, índice FAISS e API

Treina o extrator (EfficientNet-B4 + ArcFace), monta o índice vetorial e sobe a API.

⚠️ **No Colab:** *Ambiente de execução → Alterar o tipo de ambiente de execução →
GPU (T4)*. Depois, *Executar tudo*.

Roda também **localmente**: a primeira célula detecta o ambiente e usa o dataset
que já estiver em `data/`. Sem GPU funciona, só é bem mais lento.

Entrada: `data/dataset_lpf.json` + `data/images/` (do notebook 01).

| Saída | O que é |
|---|---|
| `backend/artifacts/arcface_b4.pt` | extrator treinado |
| `backend/artifacts/xiloscan.faiss` | índice vetorial das espécies |
| API em `:8000` | `/health`, `/taxonomy`, `/species`, `/identify` |

## 1 · Ambiente

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout or
      'sem GPU — vai rodar em CPU (bem mais lento, mas funciona)')

In [ ]:
import importlib.util, os, shlex, sys

PY = sys.executable
PYQ = shlex.quote(PY) if os.name != "nt" else f'"{PY}"'

faltando = [m for m in ("timm", "faiss", "cv2", "pydantic_settings", "multipart", "uvicorn")
            if importlib.util.find_spec(m) is None]
if faltando:
    print("instalando:", ", ".join(faltando))
    !{PYQ} -m pip install -q timm faiss-cpu opencv-python-headless pydantic-settings python-multipart "uvicorn[standard]" pyngrok
else:
    print("dependências já presentes")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Colab x local: mesma lógica do notebook 01.
import importlib.util
import os
from pathlib import Path

EH_COLAB = importlib.util.find_spec("google.colab") is not None

def _raiz_do_projeto(inicio: Path) -> Path | None:
    for base in [inicio, *inicio.parents]:
        if (base / "backend" / "app" / "config.py").exists() or (base / "scraper" / "lpf_scraper.py").exists():
            return base
    return None

if EH_COLAB:
    PROJETO = Path("/content/xiloscan")
else:
    PROJETO = _raiz_do_projeto(Path.cwd()) or (Path.cwd() / "xiloscan")

for sub in ("backend/app", "backend/scripts", "backend/artifacts", "data"):
    (PROJETO / sub).mkdir(parents=True, exist_ok=True)
for pkg in ("backend", "backend/app", "backend/scripts"):
    f = PROJETO / pkg / "__init__.py"
    if not f.exists():
        f.write_text("")

os.chdir(PROJETO)
sys.path.insert(0, str(PROJETO))

print("ambiente :", "Google Colab" if EH_COLAB else f"local ({sys.platform})")
print("projeto  :", PROJETO)
print("python   :", PY)

## 2 · Dataset do notebook 01

Procura `xiloscan_dataset.zip` no Drive; se não achar, pede o upload.

In [ ]:
import shutil, zipfile
from pathlib import Path

if (PROJETO / "data" / "dataset_lpf.json").exists() and (PROJETO / "data" / "images").exists():
    n = len([f for f in (PROJETO / "data" / "images").rglob("*") if f.is_file()])
    print(f"dataset já presente em {PROJETO / 'data'} — {n} imagens. Nada a extrair.")
else:
    ZIP = None
    for cand in [Path("/content/drive/MyDrive/xiloscan/xiloscan_dataset.zip"),
                 Path("/content/xiloscan_dataset.zip"),
                 PROJETO / "xiloscan_dataset.zip"]:
        if cand.exists():
            ZIP = cand
            break

    if ZIP is None and EH_COLAB:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            cand = Path("/content/drive/MyDrive/xiloscan/xiloscan_dataset.zip")
            if cand.exists():
                ZIP = cand
        except Exception:
            pass

    if ZIP is None:
        if not EH_COLAB:
            raise RuntimeError(
                "Não achei data/dataset_lpf.json nem xiloscan_dataset.zip.\n"
                "Rode antes o notebook 01 (ou refazer_dataset.bat) para gerar o dataset."
            )
        print("faça upload do xiloscan_dataset.zip gerado pelo notebook 01")
        from google.colab import files
        up = files.upload()
        ZIP = Path("/content") / next(iter(up))

    with zipfile.ZipFile(ZIP) as z:
        z.extractall("data")
    print("dataset extraído de", ZIP)

print(sorted(q.name for q in (PROJETO / "data").iterdir()))

## 3 · Código-fonte do backend

In [ ]:
%%writefile backend/app/config.py
"""Configuração central do XiloScan (12-factor via pydantic-settings)."""
from __future__ import annotations

from functools import lru_cache
from pathlib import Path

from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

ROOT = Path(__file__).resolve().parents[2]


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="XILOSCAN_", env_file=".env", extra="ignore")

    # ── dados
    dataset_path: Path = ROOT / "data" / "dataset_lpf.json"
    taxonomy_path: Path = ROOT / "data" / "mcb_taxonomy.json"
    images_root: Path = ROOT / "data" / "images"
    artifacts_dir: Path = ROOT / "backend" / "artifacts"

    # ── modelo
    backbone: str = "tf_efficientnet_b4"          # timm; alternativa: "resnet50"
    embedding_dim: int = 512
    image_size: int = 380                          # nativo da B4
    checkpoint: Path = ROOT / "backend" / "artifacts" / "arcface_b4.pt"
    device: str = "auto"                           # auto | cuda | cpu | mps

    # ── índice vetorial
    index_path: Path = ROOT / "backend" / "artifacts" / "xiloscan.faiss"
    index_meta_path: Path = ROOT / "backend" / "artifacts" / "xiloscan_index.json"
    index_type: str = "flat_ip"                    # flat_ip | ivf_pq (>50k vetores)
    top_k: int = 10

    # ── pré-processamento
    clahe_clip_limit: float = 2.5
    clahe_grid: int = 8
    center_crop_ratio: float = 0.85
    tta_enabled: bool = True                       # média de embeddings de 4 rotações

    # ── filtro híbrido
    peso_visual: float = 0.70
    peso_atributos: float = 0.30
    limiar_alta_confianca: float = 0.80
    limiar_media_confianca: float = 0.60

    # ── API
    max_upload_mb: int = 12
    cors_origins: list[str] = Field(default_factory=lambda: ["http://localhost:5173"])


@lru_cache
def get_settings() -> Settings:
    return Settings()

In [ ]:
%%writefile backend/app/schemas.py
"""Contratos da API — espelham 1:1 as interfaces TypeScript do frontend."""
from __future__ import annotations

from enum import StrEnum
from typing import Literal

from pydantic import BaseModel, Field


class NivelConfianca(StrEnum):
    ALTA = "alta"
    MEDIA = "media"
    BAIXA = "baixa"


class AnatomicalFeatures(BaseModel):
    """Descritores macroscópicos no vocabulário controlado do MCB."""
    dominio_fitogeografico: list[str] = Field(default_factory=list)
    cor_madeira: str | None = None
    aneis_crescimento: str | None = None
    vasos_porosidade: str | None = None
    vasos_agrupamento: list[str] = Field(default_factory=list)
    vasos_obstrucao: str | None = None
    tiloses_substancias: list[str] = Field(default_factory=list)
    parenquima_axial: str | None = None
    parenquima_apotraqueal: list[str] = Field(default_factory=list)
    parenquima_paratraqueal: list[str] = Field(default_factory=list)
    raios_visibilidade: list[str] = Field(default_factory=list)


class Taxonomia(BaseModel):
    nome_cientifico: str
    autor: str = ""
    familia: str = ""
    sinonimos: list[str] = Field(default_factory=list)
    nomes_populares: list[str] = Field(default_factory=list)


# seções que servem à comparação visual (vistas do lenho); as demais
# (arvore, tora, casca, madeira) são só contexto para o detalhe da espécie.
SECOES_CORTE = ("transversal", "tangencial", "radial")


class ImagemRef(BaseModel):
    """Uma foto da espécie, com a seção a que pertence."""
    tipo: str = "transversal"   # arvore|tora|casca|tangencial|radial|transversal|madeira
    url: str = ""               # servível pelo backend (/static/images/...)
    arquivo: str = ""           # caminho relativo local (images/<Especie>/<arquivo>)
    fonte_url: str = ""         # URL original no LPF


class WoodSpecies(BaseModel):
    id: str
    lpf_id: int | None = None
    fonte_url: str = ""
    taxonomia: Taxonomia
    anatomia: AnatomicalFeatures
    cor_descricao: str = ""
    gra: str = ""
    textura: str = ""
    densidade_basica: float | None = None
    imagem_url: str | None = None
    imagens: list[ImagemRef] = Field(default_factory=list)
    completeness: float = 0.0


class FeatureMatch(BaseModel):
    campo: str
    rotulo: str
    valor_usuario: list[str]
    valor_especie: list[str]
    status: Literal["coincidente", "divergente", "nao_informado"]
    peso: float


class SpeciesMatch(BaseModel):
    especie: WoodSpecies
    score_visual: float = Field(ge=0.0, le=1.0, description="similaridade de cosseno normalizada")
    score_atributos: float = Field(ge=0.0, le=1.0)
    score_final: float = Field(ge=0.0, le=1.0)
    confianca_pct: float = Field(ge=0.0, le=100.0)
    nivel: NivelConfianca
    features: list[FeatureMatch] = Field(default_factory=list)
    imagem_referencia_url: str | None = None


class ComparisonResult(BaseModel):
    request_id: str
    imagem_usuario_url: str
    processado_em_ms: int
    secao_consultada: str | None = None
    matches: list[SpeciesMatch]
    top1_ambiguo: bool = Field(
        default=False,
        description="True quando a diferença entre o 1º e o 2º score < 0.05 — exige confirmação humana.",
    )
    aviso: str = (
        "Resultado auxiliar. A identificação macroscópica de madeira exige confirmação "
        "por profissional habilitado; não use isoladamente para fins periciais ou legais."
    )


class IdentifyOptions(BaseModel):
    top_k: int = Field(default=10, ge=1, le=50)
    peso_visual: float | None = Field(default=None, ge=0.0, le=1.0)
    atributos: AnatomicalFeatures = Field(default_factory=AnatomicalFeatures)
    secao: Literal["transversal", "tangencial", "radial"] | None = Field(
        default=None,
        description="Seção da foto do usuário; restringe a comparação às imagens da mesma seção.",
    )


class HealthResponse(BaseModel):
    status: Literal["ok", "degradado"]
    modelo_carregado: bool
    indice_carregado: bool
    n_especies: int
    n_vetores: int
    device: str
    backbone: str

In [ ]:
%%writefile backend/app/catalog.py
"""Carrega o dataset_lpf.json e expõe as espécies como WoodSpecies."""
from __future__ import annotations

import json
from pathlib import Path

from .schemas import SECOES_CORTE, AnatomicalFeatures, ImagemRef, Taxonomia, WoodSpecies

_ANAT_FIELDS = set(AnatomicalFeatures.model_fields)


def _url_estatica(arquivo: str | None) -> str:
    """caminho relativo do dataset (images/<Esp>/<arq>) -> URL servida pelo backend."""
    if not arquivo:
        return ""
    return "/static/" + arquivo.lstrip("/")


# O scraper grava o rótulo longo do LPF ("corte_transversal_macroscopico"); o
# restante do backend fala o vocabulário curto de SECOES_CORTE. A tradução vive
# aqui, na fronteira: sem ela, nenhuma foto é reconhecida como seção de corte,
# o índice sai vazio e a galeria perde a ordenação.
_ALIAS_SECAO = {
    "corte_transversal_macroscopico": "transversal",
    "corte_tangencial_macroscopico": "tangencial",
    "corte_radial_macroscopico": "radial",
    "macroscopico": "transversal",
    "transversal_macroscopico": "transversal",
}


def normaliza_secao(tipo: str | None) -> str:
    """Rótulo de imagem do dataset -> vocabulário de seções do backend."""
    if not tipo:
        return "transversal"
    t = tipo.strip().lower()
    if t in _ALIAS_SECAO:
        return _ALIAS_SECAO[t]
    # Tolera variações não mapeadas: "corte_radial_xyz" ainda contém "radial".
    for secao in SECOES_CORTE:
        if secao in t:
            return secao
    return t


def _galeria(imagens_raw: list[dict]) -> list[ImagemRef]:
    refs: list[ImagemRef] = []
    for im in imagens_raw:
        arq = im.get("arquivo")
        if not arq:
            continue  # imagem que não foi baixada (404 no LPF)
        refs.append(ImagemRef(
            tipo=normaliza_secao(im.get("tipo")),
            url=_url_estatica(arq),
            arquivo=arq,
            fonte_url=im.get("url", ""),
        ))
    # ordena: seções de corte primeiro (transversal, tangencial, radial), depois contexto
    ordem = {t: i for i, t in enumerate(
        ["transversal", "tangencial", "radial", "arvore", "tora", "casca", "madeira"]
    )}
    refs.sort(key=lambda r: ordem.get(r.tipo, 99))
    return refs


def _miniatura(galeria: list[ImagemRef]) -> str | None:
    """thumbnail representativo: prioriza a superfície do lenho, nunca a foto da árvore."""
    for secao in ("transversal", "tangencial", "radial"):
        for r in galeria:
            if r.tipo == secao:
                return r.url
    return galeria[0].url if galeria else None


class Catalog:
    def __init__(self, especies: dict[str, WoodSpecies], meta: dict):
        self._especies = especies
        self.meta = meta

    @classmethod
    def load(cls, path: Path) -> Catalog:
        raw = json.loads(Path(path).read_text(encoding="utf-8"))
        especies: dict[str, WoodSpecies] = {}
        for e in raw.get("especies", []):
            anat_src = dict(e.get("anatomia_macroscopica", {}))
            anat_src["dominio_fitogeografico"] = (
                e.get("procedencia", {}).get("dominio_fitogeografico") or []
            )
            anat = AnatomicalFeatures(
                **{k: v for k, v in anat_src.items() if k in _ANAT_FIELDS}
            )
            galeria = _galeria(e.get("imagens") or [])
            atalhos = e.get("propriedades_chave") or {}
            especies[e["id"]] = WoodSpecies(
                id=e["id"],
                lpf_id=e.get("lpf_id"),
                fonte_url=e.get("fonte_url", ""),
                taxonomia=Taxonomia(**{
                    k: v for k, v in e.get("taxonomia", {}).items()
                    if k in Taxonomia.model_fields
                }),
                anatomia=anat,
                cor_descricao=e.get("caracteres_gerais", {}).get("cor_descricao", ""),
                gra=e.get("caracteres_gerais", {}).get("gra", ""),
                textura=e.get("caracteres_gerais", {}).get("textura", ""),
                densidade_basica=atalhos.get("densidade_basica"),
                imagem_url=_miniatura(galeria),
                imagens=galeria,
                completeness=float(e.get("completeness", 0.0)),
            )
        return cls(especies, raw.get("meta", {}))

    def get(self, species_id: str) -> WoodSpecies | None:
        return self._especies.get(species_id)

    def all(self) -> list[WoodSpecies]:
        return list(self._especies.values())

    def __len__(self) -> int:
        return len(self._especies)

In [ ]:
%%writefile backend/app/preprocessing.py
"""
Pré-processamento macroscópico.

Objetivo: neutralizar as três fontes de variância que mais derrubam a acurácia
em fotos de campo de corte transversal — iluminação, escala/zoom e
enquadramento — antes de extrair o embedding.

Pipeline:
  1. EXIF-transpose e conversão para RGB
  2. corte central quadrado (remove borda/dedos/fundo)
  3. CLAHE no canal L do LAB (equalização local; preserva cromaticidade)
  4. gray-world white balance (neutraliza dominante amarela de lâmpada)
  5. resize com Lanczos para image_size
  6. normalização ImageNet
"""
from __future__ import annotations

import io

import cv2
import numpy as np
import torch
from PIL import Image, ImageOps

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_rgb(data: bytes | str) -> np.ndarray:
    """Bytes ou caminho -> ndarray RGB uint8, com EXIF já aplicado."""
    if isinstance(data, bytes):
        img = Image.open(io.BytesIO(data))
    else:
        img = Image.open(data)
    img = ImageOps.exif_transpose(img).convert("RGB")
    return np.asarray(img)


def center_crop(img: np.ndarray, ratio: float = 0.85) -> np.ndarray:
    """Corte central quadrado cobrindo `ratio` da menor dimensão."""
    h, w = img.shape[:2]
    side = int(min(h, w) * ratio)
    y0 = (h - side) // 2
    x0 = (w - side) // 2
    return img[y0 : y0 + side, x0 : x0 + side]


def apply_clahe(img: np.ndarray, clip_limit: float = 2.5, grid: int = 8) -> np.ndarray:
    """CLAHE apenas na luminância (LAB), preservando a cor da madeira."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    luminancia, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid, grid))
    luminancia = clahe.apply(luminancia)
    return cv2.cvtColor(cv2.merge((luminancia, a, b)), cv2.COLOR_LAB2RGB)


def gray_world_wb(img: np.ndarray) -> np.ndarray:
    """
    White balance gray-world. Crítico aqui: a COR é um atributo diagnóstico,
    e fotos sob luz quente deslocam toda a espécie para o bin 'amarela'.
    """
    f = img.astype(np.float32)
    means = f.reshape(-1, 3).mean(axis=0)
    gray = means.mean()
    if gray <= 1e-6:
        return img
    scale = gray / np.clip(means, 1e-6, None)
    return np.clip(f * scale, 0, 255).astype(np.uint8)


def preprocess(
    data: bytes | str,
    image_size: int = 380,
    clip_limit: float = 2.5,
    grid: int = 8,
    crop_ratio: float = 0.85,
    white_balance: bool = True,
) -> np.ndarray:
    """Retorna float32 CHW normalizado ImageNet."""
    img = load_rgb(data)
    img = center_crop(img, crop_ratio)
    if white_balance:
        img = gray_world_wb(img)
    img = apply_clahe(img, clip_limit, grid)
    img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LANCZOS4)
    arr = img.astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    return np.transpose(arr, (2, 0, 1))


def to_batch(arrays: list[np.ndarray]) -> torch.Tensor:
    return torch.from_numpy(np.stack(arrays)).float()


def tta_variants(chw: np.ndarray) -> list[np.ndarray]:
    """
    O corte transversal não tem orientação canônica: a mesma amostra
    fotografada girada deve cair no mesmo ponto do espaço de embeddings.
    4 rotações + flip horizontal = 8 vistas; a média L2-normalizada é o
    embedding final.
    """
    out: list[np.ndarray] = []
    hwc = np.transpose(chw, (1, 2, 0))
    for k in range(4):
        rot = np.rot90(hwc, k)
        out.append(np.ascontiguousarray(np.transpose(rot, (2, 0, 1))))
        out.append(np.ascontiguousarray(np.transpose(np.fliplr(rot), (2, 0, 1))))
    return out


def quality_report(data: bytes | str) -> dict:
    """
    Checagem barata de qualidade antes de gastar inferência.
    Devolve avisos acionáveis para o componente de captura do frontend.
    """
    img = load_rgb(data)
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brilho = float(gray.mean())
    saturados = float((gray >= 250).mean())

    avisos: list[str] = []
    if min(h, w) < 512:
        avisos.append("Resolução baixa: fotografe mais perto ou com câmera melhor (mín. 512 px).")
    if blur < 80:
        avisos.append("Imagem desfocada: apoie a amostra e toque para focar.")
    if brilho < 55:
        avisos.append("Imagem escura: aumente a iluminação difusa.")
    if brilho > 210 or saturados > 0.08:
        avisos.append("Imagem estourada: evite flash direto e reflexo.")
    return {
        "largura": w, "altura": h,
        "nitidez_laplaciana": round(blur, 1),
        "brilho_medio": round(brilho, 1),
        "pct_saturado": round(saturados, 4),
        "aceitavel": not avisos,
        "avisos": avisos,
    }

In [ ]:
%%writefile backend/app/model.py
"""
Extrator de características para metric learning.

Por que ArcFace e não softmax comum: com 157 classes e poucas imagens por
espécie, o que precisamos não é um classificador fechado, e sim um espaço
métrico onde amostras da mesma espécie fiquem próximas e o sistema aceite
espécies novas sem retreinar. ArcFace impõe margem angular aditiva na
hiperesfera, o que separa classes visualmente parecidas (o caso das
Lauraceae/Sapotaceae, que confundem softmax) melhor que Triplet Loss —
e sem a engenharia de mineração de tripletas.
"""
from __future__ import annotations

import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


def resolve_device(pref: str = "auto") -> torch.device:
    if pref != "auto":
        return torch.device(pref)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


class GeM(nn.Module):
    """
    Generalized Mean Pooling. Em textura (que é o sinal aqui, não forma),
    GeM supera o average pooling: pondera as ativações mais fortes sem
    colapsar como o max pooling.
    """

    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clamp(min=self.eps).pow(self.p)
        x = F.adaptive_avg_pool2d(x, 1)
        return x.pow(1.0 / self.p).flatten(1)


class ArcMarginProduct(nn.Module):
    """Cabeça ArcFace (Deng et al., 2019). Usada só no treino."""

    def __init__(self, in_features: int, out_features: int, scale: float = 30.0, margin: float = 0.30):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.scale = scale
        self.margin = margin
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(1.0 - cosine.pow(2))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1), 1.0)
        return self.scale * (one_hot * phi + (1.0 - one_hot) * cosine)


class XiloEmbedder(nn.Module):
    """Backbone (timm) + GeM + BNNeck + projeção linear para `embedding_dim`."""

    def __init__(
        self,
        backbone: str = "tf_efficientnet_b4",
        embedding_dim: int = 512,
        pretrained: bool = True,
        dropout: float = 0.2,
    ):
        super().__init__()
        import timm

        self.backbone_name = backbone
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.pool = GeM()
        self.dropout = nn.Dropout(dropout)
        self.neck = nn.Linear(feat_dim, embedding_dim)
        self.bn = nn.BatchNorm1d(embedding_dim)
        self.embedding_dim = embedding_dim

    def forward(self, x: torch.Tensor, normalize: bool = True) -> torch.Tensor:
        feats = self.backbone.forward_features(x)
        if feats.ndim == 3:  # backbones ViT-like: (B, N, C) -> (B, C, H, W)
            b, n, c = feats.shape
            s = int(n ** 0.5)
            feats = feats[:, -s * s :, :].transpose(1, 2).reshape(b, c, s, s)
        x = self.pool(feats)
        x = self.bn(self.neck(self.dropout(x)))
        return F.normalize(x, dim=1) if normalize else x


class XiloArcFaceModel(nn.Module):
    """Wrapper de treino: embedder + cabeça ArcFace."""

    def __init__(self, n_classes: int, backbone: str = "tf_efficientnet_b4",
                 embedding_dim: int = 512, scale: float = 30.0, margin: float = 0.30,
                 pretrained: bool = True):
        super().__init__()
        self.embedder = XiloEmbedder(backbone, embedding_dim, pretrained=pretrained)
        self.head = ArcMarginProduct(embedding_dim, n_classes, scale, margin)

    def forward(self, x: torch.Tensor, labels: torch.Tensor | None = None):
        emb = self.embedder(x)
        if labels is None:
            return emb
        return emb, self.head(emb, labels)


@torch.inference_mode()
def embed_batch(model: XiloEmbedder, batch: torch.Tensor, device: torch.device) -> torch.Tensor:
    model.eval()
    out = model(batch.to(device))
    return out.detach().cpu()


def load_embedder(checkpoint: Path | None, backbone: str, embedding_dim: int,
                  device: torch.device) -> tuple[XiloEmbedder, bool]:
    """
    Carrega o embedder. Se não houver checkpoint treinado, cai para os pesos
    ImageNet — funciona como baseline, mas a acurácia é MUITO inferior:
    trate `treinado=False` como modo de desenvolvimento.
    """
    treinado = bool(checkpoint and Path(checkpoint).exists())
    model = XiloEmbedder(backbone, embedding_dim, pretrained=not treinado)
    if treinado:
        state = torch.load(checkpoint, map_location="cpu")
        state = state.get("embedder", state.get("model", state))
        model.load_state_dict(state, strict=False)
    return model.to(device).eval(), treinado

In [ ]:
%%writefile backend/app/index.py
"""
Banco vetorial FAISS.

Escolha: IndexFlatIP sobre vetores L2-normalizados == similaridade de cosseno
exata. Com 157 espécies × ~5-40 imagens, são no máximo alguns milhares de
vetores — busca exata custa microssegundos e evita a perda de recall do
IVF/PQ. O caminho `ivf_pq` fica pronto para quando o acervo crescer com
fotos de campo dos usuários.

Agregação: cada espécie tem N vetores (imagem oficial + augmentations/fotos).
O score da espécie é o MÁXIMO entre seus vetores, não a média — uma amostra
que casa perfeitamente com uma das vistas não deve ser penalizada pelas outras.
"""
from __future__ import annotations

import json
import logging
from pathlib import Path

import faiss
import numpy as np

log = logging.getLogger("xiloscan.index")

_SECOES = ("transversal", "tangencial", "radial", "arvore", "tora", "casca", "madeira")


def _tipo_de_path(caminho: str) -> str:
    """deriva a seção do nome do arquivo <especie>_<tipo>_NN.jpg (fallback)."""
    nome = Path(caminho).stem.lower()
    for secao in _SECOES:
        if f"_{secao}_" in nome or nome.endswith(f"_{secao}"):
            return secao
    return "transversal"


class VectorIndex:
    def __init__(self, dim: int, index_type: str = "flat_ip"):
        self.dim = dim
        self.index_type = index_type
        self.index: faiss.Index | None = None
        self.species_ids: list[str] = []      # species_id por vetor
        self.image_paths: list[str] = []      # imagem de origem por vetor
        self.tipos: list[str] = []            # seção (transversal/tangencial/...) por vetor

    # ────────────────────────────────────────────────────────────── build

    def build(
        self,
        vectors: np.ndarray,
        species_ids: list[str],
        image_paths: list[str],
        tipos: list[str] | None = None,
    ) -> None:
        if vectors.ndim != 2 or vectors.shape[1] != self.dim:
            raise ValueError(f"esperado (N,{self.dim}), recebido {vectors.shape}")
        if len(species_ids) != len(vectors):
            raise ValueError("species_ids e vectors com tamanhos diferentes")
        if tipos is None:
            tipos = [_tipo_de_path(p) for p in image_paths]

        vectors = np.ascontiguousarray(vectors.astype(np.float32))
        faiss.normalize_L2(vectors)

        n = len(vectors)
        if self.index_type == "ivf_pq" and n >= 4096:
            nlist = min(int(4 * np.sqrt(n)), n // 39)
            quantizer = faiss.IndexFlatIP(self.dim)
            idx = faiss.IndexIVFPQ(quantizer, self.dim, nlist, 64, 8, faiss.METRIC_INNER_PRODUCT)
            idx.train(vectors)
            idx.nprobe = max(8, nlist // 16)
        else:
            if self.index_type == "ivf_pq":
                log.warning("poucos vetores (%d) para IVF-PQ; usando IndexFlatIP", n)
            idx = faiss.IndexFlatIP(self.dim)

        idx.add(vectors)
        self.index = idx
        self.species_ids = list(species_ids)
        self.image_paths = list(image_paths)
        self.tipos = list(tipos)
        log.info("índice construído: %d vetores, %d espécies", n, len(set(species_ids)))

    # ─────────────────────────────────────────────────────────────── search

    def search_species(
        self,
        query: np.ndarray,
        top_k: int = 10,
        secoes: set[str] | None = None,
    ) -> list[tuple[str, float, str]]:
        """
        Busca e agrega por espécie (max-pooling dos scores).
        Retorna [(species_id, score_cosseno_0a1, imagem_mais_parecida)].

        Se `secoes` for dado, só vetores dessas seções (ex.: {'transversal'})
        entram na comparação — uma foto transversal casa só com transversais.
        """
        if self.index is None:
            raise RuntimeError("índice não construído/carregado")
        q = np.ascontiguousarray(query.astype(np.float32).reshape(1, -1))
        faiss.normalize_L2(q)

        filtrar = bool(secoes) and bool(self.tipos)
        # com filtro, varre tudo para não perder vetores da seção; senão, busca ampla
        if filtrar:
            k = len(self.species_ids)
        else:
            k = min(len(self.species_ids), max(top_k * 12, 64))
        scores, idxs = self.index.search(q, k)

        melhor: dict[str, tuple[float, str]] = {}
        for score, i in zip(scores[0], idxs[0], strict=False):
            if i < 0:
                continue
            if filtrar and self.tipos[i] not in secoes:
                continue
            sid = self.species_ids[i]
            # cosseno ∈ [-1,1] -> [0,1]
            s = float((score + 1.0) / 2.0)
            if sid not in melhor or s > melhor[sid][0]:
                melhor[sid] = (s, self.image_paths[i])

        ordenado = sorted(melhor.items(), key=lambda kv: kv[1][0], reverse=True)
        return [(sid, s, img) for sid, (s, img) in ordenado[:top_k]]

    # ────────────────────────────────────────────────────────── persistência

    def save(self, index_path: Path, meta_path: Path) -> None:
        if self.index is None:
            raise RuntimeError("nada para salvar")
        index_path.parent.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(index_path))
        meta_path.write_text(
            json.dumps(
                {
                    "dim": self.dim,
                    "index_type": self.index_type,
                    "n_vectors": len(self.species_ids),
                    "species_ids": self.species_ids,
                    "image_paths": self.image_paths,
                    "tipos": self.tipos,
                },
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        log.info("índice salvo em %s", index_path)

    @classmethod
    def load(cls, index_path: Path, meta_path: Path) -> VectorIndex:
        meta = json.loads(Path(meta_path).read_text(encoding="utf-8"))
        obj = cls(meta["dim"], meta.get("index_type", "flat_ip"))
        obj.index = faiss.read_index(str(index_path))
        obj.species_ids = meta["species_ids"]
        obj.image_paths = meta["image_paths"]
        obj.tipos = meta.get("tipos") or [_tipo_de_path(p) for p in obj.image_paths]
        if obj.index.ntotal != len(obj.species_ids):
            raise RuntimeError(
                f"índice inconsistente: {obj.index.ntotal} vetores vs "
                f"{len(obj.species_ids)} labels — reconstrua com build_index.py"
            )
        return obj

    @property
    def n_species(self) -> int:
        return len(set(self.species_ids))

    @property
    def n_vectors(self) -> int:
        return len(self.species_ids)

In [ ]:
%%writefile backend/app/hybrid.py
"""
Filtro híbrido hierárquico: similaridade visual × atributos macroscópicos.

Racional: o embedding sozinho confunde espécies de textura parecida, e os
atributos sozinhos não discriminam (dezenas de espécies compartilham
"difusa + vasicêntrico + amarela"). Combinando os dois, o atributo age como
um PRIOR que reordena o ranking visual — sem eliminar candidatos, porque o
usuário erra ao informar atributos e uma eliminação dura esconderia a
espécie correta.

score_final = w_visual * score_visual + w_attr * score_atributos

score_atributos é a média ponderada de concordâncias por campo (pesos vindos
de mcb_taxonomy.json). Campo não informado pelo usuário é NEUTRO: sai do
denominador em vez de contar como divergência. Se o usuário não informar
nada, score_atributos = score_visual (o híbrido vira o visual puro).
"""
from __future__ import annotations

import json
from pathlib import Path

from .schemas import AnatomicalFeatures, FeatureMatch, NivelConfianca

CAMPOS_MULTI = {
    "dominio_fitogeografico", "vasos_agrupamento", "tiloses_substancias",
    "parenquima_apotraqueal", "parenquima_paratraqueal", "raios_visibilidade",
}


class HybridScorer:
    def __init__(self, taxonomy_path: Path, peso_visual: float = 0.70,
                 limiar_alta: float = 0.80, limiar_media: float = 0.60):
        tax = json.loads(Path(taxonomy_path).read_text(encoding="utf-8"))
        self.pesos: dict[str, float] = {}
        self.rotulos: dict[str, str] = {}
        for cat in tax["categorias"]:
            self.pesos[cat["chave"]] = float(cat.get("peso_filtro_hibrido", 0.1))
            self.rotulos[cat["chave"]] = cat["rotulo"]
        self.peso_visual = peso_visual
        self.peso_atributos = 1.0 - peso_visual
        self.limiar_alta = limiar_alta
        self.limiar_media = limiar_media

    @staticmethod
    def _as_list(v) -> list[str]:
        if v is None:
            return []
        return [x for x in (v if isinstance(v, list) else [v]) if x]

    def compare_features(
        self, usuario: AnatomicalFeatures, especie: AnatomicalFeatures
    ) -> tuple[float, list[FeatureMatch]]:
        detalhes: list[FeatureMatch] = []
        soma_peso = 0.0
        soma_score = 0.0

        for campo, peso in self.pesos.items():
            vu = self._as_list(getattr(usuario, campo, None))
            ve = self._as_list(getattr(especie, campo, None))

            if not vu:
                # usuário não informou -> neutro, não entra no denominador
                detalhes.append(FeatureMatch(
                    campo=campo, rotulo=self.rotulos[campo], valor_usuario=[],
                    valor_especie=ve, status="nao_informado", peso=peso))
                continue

            if not ve:
                # a espécie não tem o dado preenchido (lacuna do overlay MCB):
                # também é neutro — não punimos o usuário por lacuna nossa
                detalhes.append(FeatureMatch(
                    campo=campo, rotulo=self.rotulos[campo], valor_usuario=vu,
                    valor_especie=[], status="nao_informado", peso=peso))
                continue

            if campo in CAMPOS_MULTI:
                inter = set(vu) & set(ve)
                s = len(inter) / len(set(vu))          # cobertura do que o usuário viu
            else:
                s = 1.0 if vu[0] == ve[0] else 0.0

            soma_peso += peso
            soma_score += peso * s
            detalhes.append(FeatureMatch(
                campo=campo, rotulo=self.rotulos[campo], valor_usuario=vu,
                valor_especie=ve, status="coincidente" if s > 0 else "divergente",
                peso=peso))

        score = soma_score / soma_peso if soma_peso > 0 else 0.0
        return score, detalhes

    def combine(self, score_visual: float, score_atributos: float,
                usuario_informou: bool, peso_visual: float | None = None) -> float:
        if not usuario_informou:
            return score_visual
        wv = self.peso_visual if peso_visual is None else peso_visual
        return wv * score_visual + (1.0 - wv) * score_atributos

    def nivel(self, score_final: float) -> NivelConfianca:
        if score_final >= self.limiar_alta:
            return NivelConfianca.ALTA
        if score_final >= self.limiar_media:
            return NivelConfianca.MEDIA
        return NivelConfianca.BAIXA

    @staticmethod
    def usuario_informou(a: AnatomicalFeatures) -> bool:
        return any(
            (v if isinstance(v, list) else [v] if v else [])
            for v in a.model_dump().values()
        )

    @staticmethod
    def top1_ambiguo(scores: list[float], delta: float = 0.05) -> bool:
        return len(scores) >= 2 and (scores[0] - scores[1]) < delta

In [ ]:
%%writefile backend/app/main.py
"""
XiloScan API — FastAPI.

Rotas:
  GET  /health
  GET  /taxonomy               vocabulário controlado para montar os filtros no frontend
  GET  /species                catálogo paginado
  GET  /species/{id}
  POST /identify               imagem (+ atributos opcionais) -> ComparisonResult
  POST /quality-check          checagem barata de foco/luz antes de identificar
"""
from __future__ import annotations

import json
import logging
import os
import time
import uuid
from contextlib import asynccontextmanager
from pathlib import Path

import numpy as np
import torch
from fastapi import Depends, FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles

from .catalog import Catalog
from .config import Settings, get_settings
from .hybrid import HybridScorer
from .index import VectorIndex
from .model import load_embedder, resolve_device
from .preprocessing import preprocess, quality_report, to_batch, tta_variants
from .schemas import (
    ComparisonResult,
    HealthResponse,
    IdentifyOptions,
    SpeciesMatch,
)

log = logging.getLogger("xiloscan.api")
STATE: dict = {}

# Cadastro (Railway/Postgres). Opcional: no Colab, sem SQLAlchemy, a API de
# identificação roda igual — só não expõe /auth.
try:
    from .auth import init_db as _init_auth_db
    from .auth import router as _auth_router
    _AUTH_OK = True
except Exception as _exc:  # noqa: BLE001
    _AUTH_OK = False
    logging.getLogger("xiloscan.api").warning("auth desativado (%s)", _exc)


@asynccontextmanager
async def lifespan(app: FastAPI):
    s = get_settings()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s")
    device = resolve_device(s.device)

    STATE["device"] = device
    STATE["catalog"] = Catalog.load(s.dataset_path) if s.dataset_path.exists() else Catalog({}, {})
    STATE["scorer"] = HybridScorer(
        s.taxonomy_path, s.peso_visual, s.limiar_alta_confianca, s.limiar_media_confianca
    )

    try:
        model, treinado = load_embedder(s.checkpoint, s.backbone, s.embedding_dim, device)
        STATE["model"] = model
        STATE["treinado"] = treinado
        if not treinado:
            log.warning(
                "checkpoint ausente (%s) — rodando com pesos ImageNet. "
                "Acurácia de desenvolvimento apenas: treine com scripts/train.py",
                s.checkpoint,
            )
    except Exception:
        log.exception("falha ao carregar o modelo")
        STATE["model"] = None
        STATE["treinado"] = False

    try:
        STATE["index"] = VectorIndex.load(s.index_path, s.index_meta_path)
    except Exception as exc:
        log.warning("índice FAISS indisponível (%s) — rode scripts/build_index.py", exc)
        STATE["index"] = None

    if _AUTH_OK:
        try:
            _init_auth_db()
            log.info("cadastro (auth) ativo")
        except Exception:  # noqa: BLE001
            log.exception("falha ao inicializar o banco de cadastro")

    log.info("XiloScan pronto: %d espécies | device=%s", len(STATE["catalog"]), device)
    yield
    STATE.clear()


app = FastAPI(
    title="XiloScan API",
    version="1.0.0",
    description="Identificação de madeiras por imagem macroscópica — catálogo LPF/SFB + vocabulário anatômico do app Madeiras Comerciais do Brasil (MCB).",
    lifespan=lifespan,
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=get_settings().cors_origins,
    allow_methods=["*"],
    allow_headers=["*"],
)

if _AUTH_OK:
    app.include_router(_auth_router)

_img_root = get_settings().images_root
if _img_root.exists():
    app.mount("/static/images", StaticFiles(directory=str(_img_root)), name="images")


# ───────────────────────────────────────────────────────────────── endpoints

@app.get("/health", response_model=HealthResponse)
def health(s: Settings = Depends(get_settings)) -> HealthResponse:
    idx: VectorIndex | None = STATE.get("index")
    ok = STATE.get("model") is not None and idx is not None and len(STATE["catalog"]) > 0
    return HealthResponse(
        status="ok" if ok else "degradado",
        modelo_carregado=STATE.get("model") is not None,
        indice_carregado=idx is not None,
        n_especies=len(STATE.get("catalog", [])),
        n_vetores=idx.n_vectors if idx else 0,
        device=str(STATE.get("device", "?")),
        backbone=s.backbone,
    )


@app.get("/taxonomy")
def taxonomy(s: Settings = Depends(get_settings)):
    return JSONResponse(json.loads(Path(s.taxonomy_path).read_text(encoding="utf-8")))


@app.get("/species")
def list_species(offset: int = 0, limit: int = 50, q: str = ""):
    todas = STATE["catalog"].all()
    if q:
        ql = q.lower()
        todas = [
            e for e in todas
            if ql in e.taxonomia.nome_cientifico.lower()
            or any(ql in n.lower() for n in e.taxonomia.nomes_populares)
        ]
    return {"total": len(todas), "items": todas[offset : offset + limit]}


@app.get("/species/{species_id}")
def get_species(species_id: str):
    esp = STATE["catalog"].get(species_id)
    if esp is None:
        raise HTTPException(404, f"espécie '{species_id}' não encontrada")
    return esp


def _ref_url(img_ref: str | None, images_root) -> str | None:
    """caminho absoluto do vetor indexado -> URL servível /static/images/<sub>/<arq>."""
    if not img_ref:
        return None
    try:
        rel = Path(img_ref).resolve().relative_to(Path(images_root).resolve())
        return f"/static/images/{rel.as_posix()}"
    except Exception:  # noqa: BLE001
        return None


async def _read_upload(file: UploadFile, max_mb: int) -> bytes:
    data = await file.read()
    if not data:
        raise HTTPException(400, "arquivo vazio")
    if len(data) > max_mb * 1024 * 1024:
        raise HTTPException(413, f"arquivo acima de {max_mb} MB")
    if not (file.content_type or "").startswith("image/"):
        raise HTTPException(415, f"tipo não suportado: {file.content_type}")
    return data


@app.post("/quality-check")
async def quality_check(file: UploadFile = File(...), s: Settings = Depends(get_settings)):
    data = await _read_upload(file, s.max_upload_mb)
    try:
        return quality_report(data)
    except Exception as exc:  # noqa: BLE001
        raise HTTPException(422, f"não foi possível ler a imagem: {exc}") from exc


@app.post("/identify", response_model=ComparisonResult)
async def identify(
    file: UploadFile = File(...),
    options: str = Form("{}"),
    s: Settings = Depends(get_settings),
):
    t0 = time.perf_counter()
    model = STATE.get("model")
    idx: VectorIndex | None = STATE.get("index")
    if model is None or idx is None:
        raise HTTPException(503, "modelo ou índice indisponível — veja /health")

    try:
        opts = IdentifyOptions.model_validate_json(options or "{}")
    except Exception as exc:  # noqa: BLE001
        raise HTTPException(422, f"campo 'options' inválido: {exc}") from exc

    data = await _read_upload(file, s.max_upload_mb)

    # 1. pré-processamento + TTA
    try:
        chw = preprocess(
            data, s.image_size, s.clahe_clip_limit, s.clahe_grid, s.center_crop_ratio
        )
    except Exception as exc:  # noqa: BLE001
        raise HTTPException(422, f"imagem ilegível: {exc}") from exc

    variantes = tta_variants(chw) if s.tta_enabled else [chw]
    batch = to_batch(variantes)

    # 2. embedding (média L2-normalizada das vistas)
    with torch.inference_mode():
        embs = model(batch.to(STATE["device"])).cpu().numpy()
    query = embs.mean(axis=0)
    query /= np.linalg.norm(query) + 1e-12

    # 3. busca vetorial (restrita à seção escolhida, se houver)
    secoes = {opts.secao} if opts.secao else None
    brutos = idx.search_species(query, top_k=max(opts.top_k * 2, 20), secoes=secoes)

    # 4. filtro híbrido
    scorer: HybridScorer = STATE["scorer"]
    informou = scorer.usuario_informou(opts.atributos)
    catalog: Catalog = STATE["catalog"]

    matches: list[SpeciesMatch] = []
    for species_id, score_visual, img_ref in brutos:
        esp = catalog.get(species_id)
        if esp is None:
            continue
        score_attr, detalhes = scorer.compare_features(opts.atributos, esp.anatomia)
        final = scorer.combine(score_visual, score_attr, informou, opts.peso_visual)
        matches.append(SpeciesMatch(
            especie=esp,
            score_visual=round(score_visual, 4),
            score_atributos=round(score_attr, 4),
            score_final=round(final, 4),
            confianca_pct=round(final * 100, 1),
            nivel=scorer.nivel(final),
            features=detalhes,
            imagem_referencia_url=_ref_url(img_ref, s.images_root) or esp.imagem_url,
        ))

    matches.sort(key=lambda m: m.score_final, reverse=True)
    matches = matches[: opts.top_k]

    return ComparisonResult(
        request_id=str(uuid.uuid4()),
        imagem_usuario_url="",  # preenchido pelo cliente (objectURL) ou por storage
        processado_em_ms=int((time.perf_counter() - t0) * 1000),
        secao_consultada=opts.secao,
        matches=matches,
        top1_ambiguo=scorer.top1_ambiguo([m.score_final for m in matches]),
    )


# ── SPA (deploy único no Railway): serve o frontend construído na mesma origem.
# Montado por ÚLTIMO, para não capturar as rotas da API. Guardado pela existência
# do dist — no Colab (identify-only) nada muda.
_frontend_dist = Path(
    os.getenv("XILOSCAN_FRONTEND_DIST", str(Path(__file__).resolve().parents[2] / "frontend" / "dist"))
)
if _frontend_dist.is_dir():
    app.mount("/", StaticFiles(directory=str(_frontend_dist), html=True), name="spa")
    log.info("SPA servido de %s", _frontend_dist)

In [ ]:
%%writefile backend/scripts/build_index.py
"""
Converte as imagens catalogadas do LPF em embeddings e monta o índice FAISS.

    python -m backend.scripts.build_index --augment 8

Como funciona:
  1. lê data/dataset_lpf.json e resolve o caminho local de cada imagem
  2. gera K vistas por imagem (rotações + flips, o mesmo TTA da inferência),
     porque a maioria das espécies tem UMA única foto oficial — sem isso o
     índice teria 1 vetor por classe e nenhuma tolerância a orientação
  3. extrai embeddings em lote e grava xiloscan.faiss + metadados
"""
from __future__ import annotations

import argparse
import logging
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from backend.app.catalog import Catalog  # noqa: E402
from backend.app.config import get_settings  # noqa: E402
from backend.app.index import VectorIndex  # noqa: E402
from backend.app.model import load_embedder, resolve_device  # noqa: E402
from backend.app.preprocessing import preprocess, to_batch, tta_variants  # noqa: E402
from backend.app.schemas import SECOES_CORTE  # noqa: E402

log = logging.getLogger("xiloscan.build_index")


def resolver_imagem(entrada: str | None, images_root: Path, data_root: Path) -> Path | None:
    if not entrada:
        return None
    p = Path(entrada)
    for cand in (p, data_root / p, images_root / p.name, data_root / "images" / p.name):
        if cand.exists() and cand.is_file():
            return cand
    return None


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--augment", type=int, default=8, choices=[1, 2, 4, 8],
                    help="vistas por imagem (8 = 4 rotações × flip)")
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--limit", type=int, default=0, help="processar só N espécies (debug)")
    args = ap.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s")

    s = get_settings()
    device = resolve_device(s.device)
    model, treinado = load_embedder(s.checkpoint, s.backbone, s.embedding_dim, device)
    if not treinado:
        log.warning("SEM checkpoint treinado — índice construído sobre features ImageNet. "
                    "Use apenas para desenvolvimento.")

    catalog = Catalog.load(s.dataset_path)
    especies = catalog.all()[: args.limit] if args.limit else catalog.all()
    log.info("%d espécies no catálogo", len(especies))

    vetores: list[np.ndarray] = []
    labels: list[str] = []
    caminhos: list[str] = []
    tipos: list[str] = []
    sem_imagem: list[str] = []

    buffer: list[np.ndarray] = []
    buf_labels: list[str] = []
    buf_paths: list[str] = []
    buf_tipos: list[str] = []

    def flush() -> None:
        if not buffer:
            return
        with torch.inference_mode():
            emb = model(to_batch(buffer).to(device)).cpu().numpy()
        vetores.append(emb)
        labels.extend(buf_labels)
        caminhos.extend(buf_paths)
        tipos.extend(buf_tipos)
        buffer.clear(); buf_labels.clear(); buf_paths.clear(); buf_tipos.clear()

    for i, esp in enumerate(especies, 1):
        # indexa só as vistas do lenho (transversal/tangencial/radial); árvore,
        # tora e casca são contexto para o detalhe, não servem à comparação.
        fotos = [im for im in esp.imagens if im.tipo in SECOES_CORTE]
        if not fotos:
            sem_imagem.append(esp.id)
            continue
        for im in fotos:
            img_path = resolver_imagem(im.arquivo, s.images_root, s.dataset_path.parent)
            if img_path is None:
                continue
            try:
                chw = preprocess(str(img_path), s.image_size, s.clahe_clip_limit,
                                 s.clahe_grid, s.center_crop_ratio)
            except Exception as exc:  # noqa: BLE001
                log.error("falha ao pré-processar %s: %s", img_path, exc)
                continue
            vistas = tta_variants(chw)[: args.augment] if args.augment > 1 else [chw]
            for v in vistas:
                buffer.append(v)
                buf_labels.append(esp.id)
                buf_paths.append(str(img_path))
                buf_tipos.append(im.tipo)
                if len(buffer) >= args.batch_size:
                    flush()
        if i % 25 == 0:
            log.info("  %d/%d espécies", i, len(especies))
    flush()

    if not vetores:
        log.error("nenhum embedding gerado — rode o scraper com download de imagens primeiro")
        return 1

    matriz = np.vstack(vetores).astype(np.float32)
    log.info("embeddings: %s | espécies com imagem: %d | sem imagem: %d",
             matriz.shape, len(set(labels)), len(sem_imagem))
    if sem_imagem:
        log.warning("sem imagem local: %s%s", sem_imagem[:12],
                    " ..." if len(sem_imagem) > 12 else "")

    index = VectorIndex(s.embedding_dim, s.index_type)
    index.build(matriz, labels, caminhos, tipos)
    index.save(s.index_path, s.index_meta_path)
    log.info("✔ índice pronto: %d vetores / %d espécies", index.n_vectors, index.n_species)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile backend/scripts/train.py
"""
Treino do extrator com ArcFace (metric learning).

    python -m backend.scripts.train --epochs 40 --batch-size 24

Pré-requisito: mais de uma imagem por espécie. Com a foto única do LPF por
espécie, o treino aprende só invariância a augmentation — útil, mas o ganho
real vem de adicionar fotos de campo por espécie em data/images/<Especie>/.

Estratégia:
  * split estratificado por espécie (train/val), val = 1 imagem/classe quando houver
  * augmentation forte de COR e ILUMINAÇÃO (é o que varia em campo) e
    geométrica leve — sem distorcer a escala anatômica dos poros
  * warmup + cosine schedule; margem ArcFace com ramp-up (0.1 -> 0.3)
    para o modelo não colapsar nas primeiras épocas
  * métrica de validação: recall@1 por busca de cosseno contra o resto do
    conjunto (o que a aplicação de fato faz), não acurácia de softmax
"""
from __future__ import annotations

import argparse
import logging
import math
import random
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from backend.app.config import get_settings  # noqa: E402
from backend.app.model import XiloArcFaceModel, resolve_device  # noqa: E402
from backend.app.preprocessing import preprocess  # noqa: E402

log = logging.getLogger("xiloscan.train")


class WoodDataset(Dataset):
    def __init__(self, samples: list[tuple[Path, int]], image_size: int, train: bool):
        self.samples = samples
        self.image_size = image_size
        self.train = train

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, i: int):
        path, label = self.samples[i]
        arr = preprocess(str(path), self.image_size)
        x = torch.from_numpy(arr).float()
        if self.train:
            k = random.randint(0, 3)
            if k:
                x = torch.rot90(x, k, dims=(1, 2))
            if random.random() < 0.5:
                x = torch.flip(x, dims=(2,))
            # jitter de brilho/contraste no espaço já normalizado
            x = x * random.uniform(0.88, 1.12) + random.uniform(-0.10, 0.10)
            if random.random() < 0.25:  # ruído sensor
                x = x + torch.randn_like(x) * 0.02
        return x, label


def coletar(images_root: Path) -> tuple[list[tuple[Path, int]], list[str]]:
    classes = sorted(p.name for p in images_root.iterdir() if p.is_dir())
    idx = {c: i for i, c in enumerate(classes)}
    amostras: list[tuple[Path, int]] = []
    for c in classes:
        for img in sorted((images_root / c).glob("*")):
            if img.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                amostras.append((img, idx[c]))
    return amostras, classes


def split(amostras, val_por_classe: int = 1):
    por_classe = defaultdict(list)
    for s in amostras:
        por_classe[s[1]].append(s)
    treino, val = [], []
    for _, itens in por_classe.items():
        random.shuffle(itens)
        if len(itens) > val_por_classe + 1:
            val += itens[:val_por_classe]
            treino += itens[val_por_classe:]
        else:
            treino += itens
    return treino, val


@torch.inference_mode()
def _embed(model, loader, device):
    model.eval()
    embs, labels = [], []
    for x, y in loader:
        embs.append(model.embedder(x.to(device)).cpu())
        labels.append(y)
    if not embs:
        return None, None
    return F.normalize(torch.cat(embs), dim=1), torch.cat(labels)


@torch.inference_mode()
def recall_at_1(model, query_loader, gallery_loader, device) -> float:
    """
    Recall@1 = para cada imagem de validação, a espécie do vizinho mais próximo
    na GALERIA DE TREINO é a correta? É exatamente o que o app faz (consulta
    contra o catálogo indexado). Comparar val-contra-val daria 0, pois cada
    espécie aparece uma única vez na validação.
    """
    Eq, yq = _embed(model, query_loader, device)
    Eg, yg = _embed(model, gallery_loader, device)
    if Eq is None or Eg is None:
        return float("nan")
    pred = yg[(Eq @ Eg.T).argmax(dim=1)]
    return (pred == yq).float().mean().item()


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch-size", type=int, default=24)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--weight-decay", type=float, default=1e-4)
    ap.add_argument("--warmup", type=int, default=3)
    ap.add_argument("--margin", type=float, default=0.30)
    ap.add_argument("--scale", type=float, default=30.0)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=2)
    args = ap.parse_args(argv)

    logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s")
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)

    s = get_settings()
    device = resolve_device(s.device)
    amostras, classes = coletar(s.images_root)
    if not amostras:
        log.error("nenhuma imagem em %s — rode o scraper primeiro", s.images_root)
        return 1

    treino, val = split(amostras)
    log.info("%d imagens | %d classes | treino=%d val=%d",
             len(amostras), len(classes), len(treino), len(val))
    if len(amostras) / max(len(classes), 1) < 2:
        log.warning("média < 2 imagens por espécie: o treino aprenderá pouco além de "
                    "invariância a augmentation. Adicione fotos de campo por espécie.")

    dl_tr = DataLoader(WoodDataset(treino, s.image_size, True), batch_size=args.batch_size,
                       shuffle=True, num_workers=args.workers, drop_last=len(treino) > args.batch_size)
    dl_va = DataLoader(WoodDataset(val, s.image_size, False), batch_size=args.batch_size,
                       num_workers=args.workers) if val else None
    # galeria de treino SEM augmentation, usada como catálogo na métrica de recall
    dl_ga = DataLoader(WoodDataset(treino, s.image_size, False), batch_size=args.batch_size,
                       num_workers=args.workers) if val else None

    model = XiloArcFaceModel(len(classes), s.backbone, s.embedding_dim,
                             args.scale, args.margin).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    steps = max(len(dl_tr), 1)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    melhor = -1.0
    melhor_epoca = 0
    historico: list[dict] = []
    s.artifacts_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, args.epochs + 1):
        # ramp-up da margem: margem cheia desde a época 1 costuma colapsar
        model.head.margin = args.margin * min(1.0, epoch / max(args.warmup * 2, 1))
        model.train()
        perda_total = 0.0
        for i, (x, y) in enumerate(dl_tr):
            # warmup linear + cosine decay
            t = (epoch - 1) * steps + i
            total = args.epochs * steps
            warm = args.warmup * steps
            lr = args.lr * (t / max(warm, 1) if t < warm else
                            0.5 * (1 + math.cos(math.pi * (t - warm) / max(total - warm, 1))))
            for g in opt.param_groups:
                g["lr"] = lr

            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                _, logits = model(x, y)
                loss = F.cross_entropy(logits, y, label_smoothing=0.1)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            perda_total += loss.item()

        loss_media = perda_total / steps
        r1 = recall_at_1(model, dl_va, dl_ga, device) if dl_va else float("nan")
        log.info("época %2d/%d | loss %.4f | recall@1 %.4f | margem %.3f",
                 epoch, args.epochs, loss_media, r1, model.head.margin)
        historico.append({
            "epoca": epoch, "loss": round(loss_media, 4),
            "recall_at_1": None if math.isnan(r1) else round(r1, 4),
            "margem": round(model.head.margin, 3),
        })

        # '>=' guarda o modelo MAIS TREINADO em caso de empate (val é pequena);
        # com a métrica corrigida, o recall sobe e o melhor tende à última época.
        if not math.isnan(r1) and r1 >= melhor:
            melhor, melhor_epoca = r1, epoch
            torch.save({"embedder": model.embedder.state_dict(),
                        "classes": classes, "recall_at_1": r1,
                        "backbone": s.backbone, "embedding_dim": s.embedding_dim},
                       s.checkpoint)
            log.info("  ↳ checkpoint salvo (recall@1 %.4f)", r1)

    if melhor < 0:  # sem conjunto de validação: salva a última época
        torch.save({"embedder": model.embedder.state_dict(), "classes": classes,
                    "backbone": s.backbone, "embedding_dim": s.embedding_dim}, s.checkpoint)

    # relatório do treino (para conferência)
    relatorio = {
        "config": {"epochs": args.epochs, "batch_size": args.batch_size, "lr": args.lr,
                   "margin": args.margin, "scale": args.scale, "backbone": s.backbone,
                   "embedding_dim": s.embedding_dim, "device": device.type},
        "dataset": {"imagens": len(amostras), "classes": len(classes),
                    "treino": len(treino), "val": len(val),
                    "obs": "recall@1 medido nas espécies com >=3 fotos (as que têm imagem de validação)"},
        "melhor_recall_at_1": None if melhor < 0 else round(melhor, 4),
        "melhor_epoca": melhor_epoca,
        "historico": historico,
    }
    (s.artifacts_dir / "train_report.json").write_text(
        __import__("json").dumps(relatorio, ensure_ascii=False, indent=2), encoding="utf-8")

    log.info("✔ treino concluído — melhor recall@1: %.4f (época %d) | %s",
             melhor, melhor_epoca, s.checkpoint)
    log.info("relatório: %s", s.artifacts_dir / "train_report.json")
    log.info("Próximo passo: python -m backend.scripts.build_index")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
from backend.app.config import get_settings
from backend.app.catalog import Catalog

s = get_settings()
cat = Catalog.load(s.dataset_path)
print('backbone :', s.backbone, '| dim:', s.embedding_dim, '| img:', s.image_size)
print('espécies :', len(cat))
print('imagens  :', s.images_root, '->', s.images_root.exists())

## 4 · Quantas imagens por espécie?

Esta célula decide o que faz sentido treinar. O LPF publica ~1 imagem por espécie;
com isso, o ArcFace aprende basicamente invariância a augmentation.

In [ ]:
import collections
from pathlib import Path

por_esp = {p.name: len([f for f in p.glob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
           for p in sorted(Path('data/images').iterdir()) if p.is_dir()}
dist = collections.Counter(por_esp.values())
media = sum(por_esp.values()) / max(len(por_esp), 1)

print('espécies com pasta :', len(por_esp))
print('imagens totais     :', sum(por_esp.values()))
print('média por espécie  : %.2f' % media)
print('distribuição       :', dict(sorted(dist.items())))

TREINAR = media >= 2.0
print('\n' + ('✅ dá para treinar ArcFace — siga para a seção 5'
               if TREINAR else
               '⚠️  menos de 2 imagens/espécie: PULE a seção 5 e vá direto para a 6.\n'
               '    O índice sobre features ImageNet + TTA já funciona para desenvolver.\n'
               '    Ganho real vem de adicionar fotos de campo em data/images/<Especie>/.'))

## 5 · Treino ArcFace

*Pule esta seção se a célula acima disse para pular.* ~40 épocas numa T4 levam
poucos minutos com este volume de imagens.

In [ ]:
!{PYQ} -m backend.scripts.train --epochs 40 --batch-size 24 --lr 3e-4

In [ ]:
import torch
from backend.app.config import get_settings

s = get_settings()
if s.checkpoint.exists():
    ck = torch.load(s.checkpoint, map_location='cpu')
    print('backbone :', ck.get('backbone'))
    print('classes  :', len(ck.get('classes', [])))
    print('recall@1 :', ck.get('recall_at_1'))
else:
    print('sem checkpoint — o índice usará features ImageNet (esperado se você pulou a seção 5)')

## 6 · Índice FAISS

In [ ]:
!{PYQ} -m backend.scripts.build_index --augment 8 --batch-size 16

In [ ]:
# sanidade: cada imagem indexada deve recuperar a própria espécie em 1º lugar
import torch
from backend.app.config import get_settings
from backend.app.index import VectorIndex
from backend.app.model import load_embedder, resolve_device
from backend.app.preprocessing import preprocess, to_batch

s = get_settings()
dev = resolve_device(s.device)
model, treinado = load_embedder(s.checkpoint, s.backbone, s.embedding_dim, dev)
idx = VectorIndex.load(s.index_path, s.index_meta_path)
print('checkpoint treinado:', treinado, '| vetores:', idx.n_vectors, '| espécies:', idx.n_species)

# pares únicos (especie, imagem) — fromkeys sobre tuplas, sem .items()
pares = list(dict.fromkeys(zip(idx.species_ids, idx.image_paths)))[:60]
acertos = 0
for sid, img in pares:
    with torch.inference_mode():
        q = model(to_batch([preprocess(img, s.image_size)]).to(dev)).cpu().numpy()[0]
    acertos += idx.search_species(q, top_k=1)[0][0] == sid
print(f'recall@1 do próprio índice: {acertos}/{len(pares)} = {acertos/len(pares):.1%}')
print('(deve ficar perto de 100% — é a imagem que gerou o vetor. Bem abaixo disso = índice inconsistente.)')

## 7 · API

In [ ]:
import threading, time, requests, uvicorn
from backend.app.main import app

threading.Thread(
    target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'),
    daemon=True,
).start()

for _ in range(30):
    try:
        h = requests.get('http://127.0.0.1:8000/health', timeout=2).json()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('a API não subiu')

print(h)

In [ ]:
# Identifique uma foto. No Colab abre o seletor; localmente, aponte um caminho.
import json, requests
from pathlib import Path

FOTO_LOCAL = ""   # ex.: r"D:\\fotos\\amostra.jpg"  (deixe vazio no Colab)

if EH_COLAB and not FOTO_LOCAL:
    from google.colab import files
    up = files.upload()
    nome = next(iter(up))
    conteudo = up[nome]
else:
    caminho = Path(FOTO_LOCAL)
    if not caminho.exists():
        raise RuntimeError("Preencha FOTO_LOCAL com o caminho de uma imagem macroscópica.")
    nome, conteudo = caminho.name, caminho.read_bytes()

# opcional: atributos macroscópicos que você observou.
# códigos válidos: GET /taxonomy, ou data/mcb_taxonomy.json
opts = {"top_k": 5, "atributos": {}}   # ex.: {"cor_madeira": "AMARELA", "vasos_porosidade": "DIFUSA"}

r = requests.post("http://127.0.0.1:8000/identify",
                  files={"file": (nome, conteudo, "image/jpeg")},
                  data={"options": json.dumps(opts)}).json()

print(f"{r['processado_em_ms']} ms | resultado ambíguo: {r['top1_ambiguo']}\n")
for k, m in enumerate(r["matches"], 1):
    t = m["especie"]["taxonomia"]
    print(f"{k}. {t['nome_cientifico']:<34} {m['confianca_pct']:5.1f}%  "
          f"[{m['nivel']}]  visual {m['score_visual']:.3f} · atrib {m['score_atributos']:.3f}")
    if t["nomes_populares"]:
        print(f"   {', '.join(t['nomes_populares'][:4])}")

## 8 · Expor a API para o frontend local

Cole a URL em `frontend/.env` como `VITE_API_URL` e rode `npm run dev` na sua máquina.
Precisa de um token gratuito em <https://dashboard.ngrok.com>.

In [ ]:
from pyngrok import ngrok

# ngrok.set_auth_token('SEU_TOKEN_AQUI')
tunel = ngrok.connect(8000)
print('VITE_API_URL=' + tunel.public_url)

In [ ]:
# Salvar checkpoint e índice.
import shutil
from pathlib import Path

art = PROJETO / "backend" / "artifacts"
arquivos = [f for f in art.glob("*") if f.is_file()]
print(f"{len(arquivos)} artefato(s) em {art}:")
for f in arquivos:
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

if not EH_COLAB:
    print("\nExecução local — já estão salvos na sua pasta, nada a fazer.")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        destino = Path("/content/drive/MyDrive/xiloscan/artifacts")
        destino.mkdir(parents=True, exist_ok=True)
        for f in arquivos:
            shutil.copy(f, destino / f.name)
        print("\ncopiados para", destino)
    except Exception as exc:
        print("\nDrive indisponível:", exc, "— baixando um zip")
        shutil.make_archive("/content/xiloscan_artifacts", "zip", art)
        from google.colab import files
        files.download("/content/xiloscan_artifacts.zip")